# Multi-agent report generator

One agent that browses freely writes a report you cannot audit. This notebook builds a small report factory in LangGraph: a scoper, specialist researchers, a verifier, a writer with no tools, and a citation audit written in plain code. Its subject is your own results: the capability report and the RAGAS scores.

## Learn | Create | Grow

### Learn
Specialist workers behind delegate tools, typed handoff contracts, a supervisor, and a graph that scopes, researches, verifies, writes, and audits citations.


### Create
A report on your own results, generated by the graph over your capability report and RAGAS scores, with its source ledger and citation audit.


### Grow
Production multi-agent systems keep the source ledger and the audit next to every report. Show your team one worker trace that changed the final answer.


**Estimated time:** 50 minutes
**Reads:** capability_report, ragas_scores, corpus
**Writes:** multi_agent_report

## Setup

One chat model for every actor. Web research runs only when `TAVILY_API_KEY` is set; without it the factory works from your own results and corpus, which is the normal case. `create_agent` owns each actor's loop; LangGraph owns the lifecycle.

In [ ]:
import json, re, textwrap
from collections import Counter
from datetime import date
from typing import Literal, TypedDict

from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_core.messages import ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

from helpers.config import KEY, LLM_BASE, LLM_MODEL, TAVILY_KEY, require, budget
from helpers import workspace as ws
from helpers.llm import chat_model

require("OPENAI_API_KEY")
TODAY = date.today().isoformat()
llm = chat_model()

CAPABILITY_REPORT = ws.load("capability_report")
RAGAS = ws.load("ragas_scores")
CORPUS_DIR = ws.load_path("corpus")
PAGES = {str(p.relative_to(CORPUS_DIR)): p.read_text(encoding="utf-8") for p in ws.load("corpus") if p.suffix == ".md"}
WEB = bool(TAVILY_KEY)
print(f"✅ model {LLM_MODEL}; capability report {len(CAPABILITY_REPORT.split())} words; {len(RAGAS)} RAGAS rows; "
      f"{len(PAGES)} corpus pages; web research {'on' if WEB else 'off'}")

You should see a ✅ line with the model, a word count above fifty, a RAGAS row count above one, a page count above five, and whether web research is on. Stop here if any count is zero: run the trajectory evals and RAGAS notebooks first, or let the seed carry them.

# Learn


## Task 1 of 5 — Give every source a stable handle

Before agents can cite anything, each piece of evidence needs a durable label. Split the capability report, the RAGAS table, and every corpus page into sections, and give each a `local://` URL. Two tools expose them: a plain keyword search and an exact fetch by URL. Every tool result carries its URL, which is what makes the audit possible later.

In [ ]:
URL_PATTERN = re.compile(r'(?:https?://|local://)[^\s\]\[()<>{}"\']+')
TOKEN_RE = re.compile(r"[a-z0-9]{3,}")


def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")[:60] or "section"


def split_sections(kind: str, page: str, text: str) -> list[dict]:
    out, title, lines = [], "overview", []

    def flush():
        body = "\n".join(lines).strip()
        if body:
            out.append({"kind": kind, "id": f"{page}/{slugify(title)}", "title": f"{page}: {title}",
                        "url": f"local://{kind}/{slugify(page)}/{slugify(title)}", "text": body})
    for line in text.splitlines():
        if line.startswith("## "):
            flush()
            title, lines = line[3:].strip(), []
        else:
            lines.append(line)
    flush()
    return out


def ragas_page(rows: list[dict]) -> str:
    cols = list(rows[0].keys())
    body = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    body += ["| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows]
    return "RAGAS scores per variant of the retrieval pipeline.\n\n" + "\n".join(body)


SECTIONS = split_sections("evals", "capability-report", CAPABILITY_REPORT)
SECTIONS.append({"kind": "evals", "id": "ragas-scores", "title": "RAGAS scores", "url": "local://evals/ragas-scores",
                 "text": ragas_page(RAGAS)})
for name, text in PAGES.items():
    SECTIONS += split_sections("corpus", name.removesuffix(".md"), text)
BY_URL = {s["url"]: s for s in SECTIONS}


def tokens(text: str) -> set[str]:
    return set(TOKEN_RE.findall(text.lower()))


def search_sections(query: str, k: int = 5, kind: str | None = None) -> list[dict]:
    q = tokens(query)
    scored = [(len(q & tokens(s["title"] + "\n" + s["text"])), s) for s in SECTIONS if kind is None or s["kind"] == kind]
    return [s for n, s in sorted(scored, key=lambda x: x[0], reverse=True) if n][:k]


def preview(text: str, limit: int = 1000) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text.strip())
    return text if len(text) <= limit else text[:limit].rstrip() + "\n..."


@tool("search_sources", description="Search the team's own results (the capability report and RAGAS scores) and the corpus. "
      "Use it before making any claim. Returns stable local:// URLs for citation.")
def search_sources(query: str) -> str:
    hits = search_sections(query)
    if not hits:
        return "No matching sections."
    return "\n\n---\n\n".join(f"[{i}] {s['title']}\nURL: {s['url']}\nExcerpt:\n{preview(s['text'])}" for i, s in enumerate(hits, 1))


@tool("get_source", description="Return the full text of one source by its exact local:// URL.")
def get_source(url: str) -> str:
    s = BY_URL.get(url.strip())
    return f"{s['title']}\nURL: {s['url']}\n\n{s['text']}" if s else f"No source at {url!r}."


print(f"{len(SECTIONS)} sections: {Counter(s['kind'] for s in SECTIONS)}")
print(search_sources.invoke({"query": "pass rate worst failure regression"})[:900])

You should see a section count with both kinds, then search hits that each carry a `local://evals/...` URL. Stop here if the evals kind is missing: the capability report has no `##` headings, so the splitter put it all in one section.

## Task 2 of 5 — Contracts and the audit

Each agent returns structured data, not prose, so the graph has something to check: tasks, findings, sources, claims, the final report, and the audit. Typed output proves the shape is valid. It does not prove the facts are true. A model can put a URL it never saw in the right field, so every finding is filtered against the URLs that appeared in that run's tool messages, and the final citation audit is plain code.

In [ ]:
SpecialistName = Literal["evals", "corpus", "web"]
SourceKind = Literal["evals", "corpus", "web_guidance", "other"]


class ReportTask(BaseModel):
    task_id: str = Field(description="Stable short identifier such as task-1")
    question: str = Field(description="Focused research question for one specialist")
    specialist: SpecialistName
    rationale: str


class ReportBrief(BaseModel):
    title: str
    objective: str
    audience: str
    scope_in: list[str]
    scope_out: list[str]
    assumptions: list[str] = Field(default_factory=list)
    tasks: list[ReportTask] = Field(min_length=2, max_length=3)


class SourceRecord(BaseModel):
    url: str
    title: str
    source_type: SourceKind
    relevant_excerpt: str


class ClaimRecord(BaseModel):
    claim: str
    source_urls: list[str] = Field(min_length=1)
    confidence: Literal["high", "medium", "low"]
    caveat: str = ""


class SpecialistFindings(BaseModel):
    task_id: str
    specialist: SpecialistName
    summary: str
    sources: list[SourceRecord] = Field(default_factory=list)
    claims: list[ClaimRecord] = Field(default_factory=list)
    risks_or_gaps: list[str] = Field(default_factory=list)
    errors: list[str] = Field(default_factory=list)


class ResearchDossier(BaseModel):
    brief_title: str
    findings: list[SpecialistFindings] = Field(default_factory=list)
    synthesis: list[str] = Field(default_factory=list)
    unresolved_gaps: list[str] = Field(default_factory=list)


class VerificationReport(BaseModel):
    approved_sources: list[SourceRecord] = Field(default_factory=list)
    rejected_sources: list[str] = Field(default_factory=list)
    verified_claims: list[ClaimRecord] = Field(default_factory=list)
    unsupported_claims: list[str] = Field(default_factory=list)
    verification_summary: str


class FinalReport(BaseModel):
    title: str
    executive_summary: str
    report_markdown: str
    citations: list[str] = Field(description="Exact URLs in numeric citation order")
    open_questions: list[str] = Field(default_factory=list)


class CitationAudit(BaseModel):
    passed: bool
    unknown_urls: list[str] = Field(default_factory=list)
    duplicate_urls: list[str] = Field(default_factory=list)
    missing_citation_numbers: list[int] = Field(default_factory=list)
    markers_without_citations: list[int] = Field(default_factory=list)
    notes: list[str] = Field(default_factory=list)


class ReportEvaluation(BaseModel):
    coverage: int = Field(ge=1, le=5)
    grounding: int = Field(ge=1, le=5)
    usefulness: int = Field(ge=1, le=5)
    citation_integrity: int = Field(ge=1, le=5)
    passed: bool
    strengths: list[str]
    improvements: list[str]


print("contracts defined:", ReportBrief.__name__, SpecialistFindings.__name__, FinalReport.__name__, CitationAudit.__name__)

In [ ]:
def normalize_url(url: str) -> str:
    return url.strip().rstrip(".,;:!?)").lower()


def urls_in_value(value) -> set[str]:
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    return {normalize_url(m) for m in URL_PATTERN.findall(text)}


def observed_tool_urls(agent_result: dict) -> set[str]:
    """Every URL that appeared in a tool message during this run."""
    seen = set()
    for m in agent_result.get("messages", []):
        if isinstance(m, ToolMessage):
            seen |= urls_in_value(m.content)
    return seen


def sanitize_findings(findings: SpecialistFindings, observed: set[str]) -> SpecialistFindings:
    """Drop sources and claims the worker did not actually see."""
    sources = [s for s in findings.sources if normalize_url(s.url) in observed]
    ok = {normalize_url(s.url) for s in sources}
    claims, dropped = [], []
    for c in findings.claims:
        traced = [u for u in c.source_urls if normalize_url(u) in ok]
        (claims.append(c.model_copy(update={"source_urls": traced})) if traced else dropped.append(f"Untraced claim removed: {c.claim}"))
    errors = list(findings.errors) + (["Sources returned, but none appeared in tool messages."] if findings.sources and not sources else [])
    return findings.model_copy(update={"sources": sources, "claims": claims, "risks_or_gaps": findings.risks_or_gaps + dropped, "errors": errors})


def sanitize_verification(report: VerificationReport, observed: set[str]) -> VerificationReport:
    approved = [s for s in report.approved_sources if normalize_url(s.url) in observed]
    ok = {normalize_url(s.url) for s in approved}
    verified, unsupported = [], list(report.unsupported_claims)
    for c in report.verified_claims:
        traced = [u for u in c.source_urls if normalize_url(u) in ok]
        (verified.append(c.model_copy(update={"source_urls": traced})) if traced else unsupported.append(f"Verifier returned untraced claim: {c.claim}"))
    return report.model_copy(update={"approved_sources": approved, "verified_claims": verified, "unsupported_claims": unsupported})


def audit_report_citations(report: FinalReport, verification: VerificationReport) -> CitationAudit:
    """Plain code: allow-listed URLs, no duplicates, markers and citations line up."""
    approved = {normalize_url(s.url) for s in verification.approved_sources}
    citations = [normalize_url(u) for u in report.citations]
    counts = Counter(citations)
    markers = {int(m) for m in re.findall(r"\[(\d+)\]", report.report_markdown)}
    expected = set(range(1, len(citations) + 1))
    notes = ["Citations listed but no numeric markers in the markdown."] if citations and not markers else []
    audit = CitationAudit(unknown_urls=[u for u in citations if u not in approved],
                          duplicate_urls=sorted(u for u, n in counts.items() if n > 1),
                          missing_citation_numbers=sorted(expected - markers),
                          markers_without_citations=sorted(m for m in markers if m not in expected),
                          notes=notes, passed=False)
    audit.passed = not (audit.unknown_urls or audit.duplicate_urls or audit.missing_citation_numbers or audit.markers_without_citations or notes)
    return audit


# No-API checks: the audit must reject a bad citation every time, before any tokens are spent.
src = SourceRecord(url=SECTIONS[0]["url"], title=SECTIONS[0]["title"], source_type="evals", relevant_excerpt="fixture")
ver = VerificationReport(approved_sources=[src], verification_summary="fixture")
good = FinalReport(title="Fixture", executive_summary="Fixture", report_markdown=f"Claim [1]\n\nSources\n[1] {src.url}", citations=[src.url])
assert audit_report_citations(good, ver).passed
bad = good.model_copy(update={"citations": [src.url, "https://example.com/nope"]})
bad_audit = audit_report_citations(bad, ver)
assert not bad_audit.passed and "https://example.com/nope" in bad_audit.unknown_urls
print("deterministic checks passed:", bad_audit.model_dump())

You should see the contract names, then `deterministic checks passed` with an audit that lists the unknown URL and a missing marker 2. Stop here if an assertion fails: the audit is the cheapest part of the factory and it has to be right before anything else runs.

### ❓ Question
The audit rejected `[99]` in the fixture without a model call. What kind of citation mistake would still pass the audit and need the verifier?

Answer:

## Task 3 of 5 — Specialists, delegates, and a supervisor

Each worker gets a narrow prompt and a narrow toolbox: the evals specialist reads your results, the corpus specialist reads what the product promised and users asked, and a web specialist joins only when Tavily is configured. To the supervisor each worker is one delegate tool with its own context and budget. The scoper turns the request into two or three independent tasks first.

In [ ]:
WORKER_MODEL_CALL_LIMIT = budget(8, 4)


def worker_middleware(tools: list) -> list:
    return [ModelCallLimitMiddleware(run_limit=WORKER_MODEL_CALL_LIMIT, exit_behavior="end")] + [
        ToolCallLimitMiddleware(tool_name=t.name, run_limit=4, exit_behavior="continue") for t in tools]


EVALS_PROMPT = f"""You are the evals specialist. Today is {TODAY}. Research only the assigned task, using search_sources and
get_source over the capability report and the RAGAS scores. Quote numbers exactly. Use source_type "evals" for every
source. Every claim URL must be a local:// URL that appeared in a tool result from this run. Preserve uncertainty.
Return exactly the SpecialistFindings schema."""

CORPUS_PROMPT = f"""You are the corpus specialist. Today is {TODAY}. Research only the assigned task, using search_sources and
get_source over the corpus: the charter, prompt outputs, and transcripts. Say what the product promised and what users
asked. Use source_type "corpus". Every claim URL must be a local:// URL that appeared in a tool result from this run.
Return exactly the SpecialistFindings schema."""

LOCAL_TOOLS = [search_sources, get_source]
SPECIALISTS = {
    "evals": create_agent(model=llm, tools=LOCAL_TOOLS, system_prompt=EVALS_PROMPT, middleware=worker_middleware(LOCAL_TOOLS),
                          response_format=SpecialistFindings, name="evals_specialist"),
    "corpus": create_agent(model=llm, tools=LOCAL_TOOLS, system_prompt=CORPUS_PROMPT, middleware=worker_middleware(LOCAL_TOOLS),
                           response_format=SpecialistFindings, name="corpus_specialist"),
}
if WEB:
    from langchain_tavily import TavilySearch, TavilyExtract
    web_tools = [TavilySearch(max_results=5, topic="general", search_depth="basic", include_answer=False, include_raw_content=False),
                 TavilyExtract(extract_depth="basic", format="markdown", chunks_per_source=3)]
    WEB_PROMPT = (f"You are the web-guidance specialist. Today is {TODAY}. Research only the assigned task with the search and "
                  "extract tools. Prefer official and well-established sources. Do not invent URLs; every source URL must "
                  "appear in a search or extraction result in this run. Use source_type \"web_guidance\". "
                  "Return exactly the SpecialistFindings schema.")
    SPECIALISTS["web"] = create_agent(model=llm, tools=web_tools, system_prompt=WEB_PROMPT, middleware=worker_middleware(web_tools),
                                      response_format=SpecialistFindings, name="web_specialist")


def run_worker(name: str, task_json: str) -> SpecialistFindings:
    """Parse the task, run the worker, keep only what it actually saw."""
    try:
        task = ReportTask.model_validate_json(task_json)
    except Exception as exc:                                     # noqa: BLE001
        return SpecialistFindings(task_id="invalid-task", specialist=name, summary="The supervisor supplied an invalid task.",
                                  errors=[f"{type(exc).__name__}: {exc}"])
    try:
        result = SPECIALISTS[name].invoke({"messages": [{"role": "user", "content": "Complete this delegated research task and return structured findings:\n" + task_json}]})
        findings = SpecialistFindings.model_validate(result["structured_response"]).model_copy(update={"task_id": task.task_id, "specialist": name})
        return sanitize_findings(findings, observed_tool_urls(result))
    except Exception as exc:                                     # noqa: BLE001
        return SpecialistFindings(task_id=task.task_id, specialist=name, summary="The delegated task did not complete.",
                                  risks_or_gaps=[task.question], errors=[f"{type(exc).__name__}: {exc}"])


@tool("delegate_evals_research", description="Delegate one ReportTask (as JSON) to the evals specialist, who reads the capability report and RAGAS scores.")
def delegate_evals_research(task_json: str) -> str:
    return run_worker("evals", task_json).model_dump_json(indent=2)


@tool("delegate_corpus_research", description="Delegate one ReportTask (as JSON) to the corpus specialist, who reads the charter, prompts, and transcripts.")
def delegate_corpus_research(task_json: str) -> str:
    return run_worker("corpus", task_json).model_dump_json(indent=2)


@tool("delegate_web_research", description="Delegate one ReportTask (as JSON) to the web-guidance specialist.")
def delegate_web_research(task_json: str) -> str:
    return run_worker("web", task_json).model_dump_json(indent=2)


DELEGATES = [delegate_evals_research, delegate_corpus_research] + ([delegate_web_research] if WEB else [])
print("specialists:", list(SPECIALISTS), "| delegate tools:", [t.name for t in DELEGATES])

In [ ]:
SCOPER_PROMPT = f"""You scope report requests about the team's own agent. Today is {TODAY}. Available specialists: {', '.join(SPECIALISTS)}.
Return a ReportBrief with one independent task per available specialist, in this order: an "evals" task on where the
agent fails and by how much; a "corpus" task on what the product promised and what users asked; and, only if "web" is
available, a "web" task on current public guidance for the top failure. Record assumptions instead of asking questions."""

scoper_agent = create_agent(model=llm, tools=[], system_prompt=SCOPER_PROMPT, response_format=ReportBrief, name="report_scoper")

SUPERVISOR_PROMPT = f"""You are the lead researcher coordinating a bounded report about the team's own agent. Today is {TODAY}.
You receive a ReportBrief. For every task call the delegate tool for its specialist ({', '.join(t.name for t in DELEGATES)}),
passing the complete ReportTask as JSON. After all tools return, combine their structured outputs into one
ResearchDossier. Preserve errors, disagreements, and gaps. Do not invent sources or URLs. Do not write the report."""

research_supervisor = create_agent(
    model=llm, tools=DELEGATES, system_prompt=SUPERVISOR_PROMPT,
    middleware=[ModelCallLimitMiddleware(run_limit=6, exit_behavior="end")]
    + [ToolCallLimitMiddleware(tool_name=t.name, run_limit=2, exit_behavior="continue") for t in DELEGATES],
    response_format=ResearchDossier, name="research_supervisor")

REQUEST = ("Write a report for the engineers who own our agent: where it fails, by how much, what the product promised, "
           "and what to change first. Use the capability report, the RAGAS scores, and the corpus.")
BRIEF = ReportBrief.model_validate(scoper_agent.invoke({"messages": [{"role": "user", "content": REQUEST}]})["structured_response"])
for t in BRIEF.tasks:
    print(f"{t.task_id:<8} {t.specialist:<7} {t.question}")

You should see the specialist names and delegate tools, then two or three scoped tasks, one per specialist, each a focused question. Stop here if a task names a specialist that is not in the list: the scoper prompt and `SpecialistName` disagree.

## Task 4 of 5 — Verify, write, audit, and evaluate as a graph

Research, verification, writing, and audit are different jobs. The verifier re-opens sources before approving claims. The writer has no tools, so it cannot introduce a fact or a URL. The audit is code; the evaluator is a model that sees the audit. LangGraph owns that order: agents decide inside a node, the graph decides what runs next.

In [ ]:
VERIFIER_PROMPT = f"""You verify a research dossier about the team's agent. Today is {TODAY}. For every local:// source, use get_source to
re-open it. Approve a source only if the URL appears in a tool result and supports the claim. Reject weak or irrelevant
sources. Return VerificationReport: keep only claims supported by approved URLs, preserve unsupported claims and gaps,
never invent replacement URLs."""

verifier_agent = create_agent(
    model=llm, tools=LOCAL_TOOLS, system_prompt=VERIFIER_PROMPT,
    middleware=[ModelCallLimitMiddleware(run_limit=budget(8, 4), exit_behavior="end"),
                ToolCallLimitMiddleware(tool_name=get_source.name, run_limit=8, exit_behavior="continue"),
                ToolCallLimitMiddleware(tool_name=search_sources.name, run_limit=3, exit_behavior="continue")],
    response_format=VerificationReport, name="source_verifier")

WRITER_PROMPT = f"""You write short engineering reports. Today is {TODAY}. Use only the verified claims, approved sources, and gaps in
the supplied VerificationReport and ReportBrief. Do not add facts or URLs. In report_markdown: write for the engineers who
own the agent; separate what fails, why, what the product promised, and what to change first; use numeric citation
markers like [1], [2]; end with a Sources section mapping every marker to its exact approved URL. The citations field
must list the exact URLs in numeric order."""

writer_agent = create_agent(model=llm, tools=[], system_prompt=WRITER_PROMPT, response_format=FinalReport, name="report_writer")

EVALUATOR_PROMPT = """You evaluate a report against its brief, its verification report, and its citation audit. Score each criterion
from 1 to 5. Pass only when every score is at least 3 and citation_integrity is 5. Reward specific numbers, a clear first
change, and honest gaps. Penalise unsupported claims and missing traceability. Return ReportEvaluation."""

evaluator_agent = create_agent(model=llm, tools=[], system_prompt=EVALUATOR_PROMPT, response_format=ReportEvaluation, name="report_evaluator")
print("verifier, writer, and evaluator ready")

In [ ]:
class ReportGraphState(TypedDict, total=False):
    query: str
    brief: ReportBrief
    dossier: ResearchDossier
    verification: VerificationReport
    report: FinalReport
    citation_audit: CitationAudit
    evaluation: ReportEvaluation
    errors: list[str]


def scope_node(state: ReportGraphState) -> dict:
    r = scoper_agent.invoke({"messages": [{"role": "user", "content": state["query"]}]})
    return {"brief": ReportBrief.model_validate(r["structured_response"])}


def research_node(state: ReportGraphState) -> dict:
    brief = state["brief"]
    try:
        r = research_supervisor.invoke({"messages": [{"role": "user", "content": "Research this brief:\n" + brief.model_dump_json(indent=2)}]})
        dossier = ResearchDossier.model_validate(r["structured_response"])
        by_id = {t.task_id: t for t in brief.tasks}
        findings = [f.model_copy(update={"specialist": by_id[f.task_id].specialist}) for f in dossier.findings if f.task_id in by_id]
        missing = [f"No findings for {t.task_id}: {t.question}" for t in brief.tasks if t.task_id not in {f.task_id for f in findings}]
        return {"dossier": dossier.model_copy(update={"brief_title": brief.title, "findings": findings,
                                                     "unresolved_gaps": dossier.unresolved_gaps + missing})}
    except Exception as exc:                                     # noqa: BLE001
        return {"dossier": ResearchDossier(brief_title=brief.title, unresolved_gaps=["The supervisor failed.", f"{type(exc).__name__}: {exc}"]),
                "errors": state.get("errors", []) + [f"research_node: {type(exc).__name__}: {exc}"]}


def verify_node(state: ReportGraphState) -> dict:
    payload = {"brief": state["brief"].model_dump(), "dossier": state["dossier"].model_dump()}
    r = verifier_agent.invoke({"messages": [{"role": "user", "content": "Verify this dossier:\n" + json.dumps(payload, indent=2)}]})
    return {"verification": sanitize_verification(VerificationReport.model_validate(r["structured_response"]), observed_tool_urls(r))}


def write_node(state: ReportGraphState) -> dict:
    payload = {"brief": state["brief"].model_dump(), "verification": state["verification"].model_dump()}
    r = writer_agent.invoke({"messages": [{"role": "user", "content": "Write the final report from this verified evidence:\n" + json.dumps(payload, indent=2)}]})
    return {"report": FinalReport.model_validate(r["structured_response"])}


def audit_node(state: ReportGraphState) -> dict:
    return {"citation_audit": audit_report_citations(state["report"], state["verification"])}


def evaluate_node(state: ReportGraphState) -> dict:
    payload = {k: state[k].model_dump() for k in ("brief", "verification", "report", "citation_audit")}
    r = evaluator_agent.invoke({"messages": [{"role": "user", "content": "Evaluate this report package:\n" + json.dumps(payload, indent=2)}]})
    return {"evaluation": ReportEvaluation.model_validate(r["structured_response"])}


builder = StateGraph(ReportGraphState)
for name, fn in [("scope", scope_node), ("research", research_node), ("verify", verify_node), ("write", write_node),
                 ("audit", audit_node), ("evaluate", evaluate_node)]:
    builder.add_node(name, fn)
builder.add_edge(START, "scope")
for a, b in [("scope", "research"), ("research", "verify"), ("verify", "write"), ("write", "audit"), ("audit", "evaluate")]:
    builder.add_edge(a, b)
builder.add_edge("evaluate", END)
report_graph = builder.compile()
print(report_graph.get_graph().draw_mermaid())

You should see the three agents ready and a mermaid diagram with six nodes in a straight line from scope to evaluate. Nothing has run yet. Stop here if compile fails: a node name in the edges does not match a node you added.

### ❓ Question
Which decisions in this factory belong to an agent, and which belong to the graph? Name one you would move.

Answer:

# Create


## Task 5 of 5 — Run it on your own results and save

Stream the run. Each line names the node that just finished and the state it wrote. Then read the report, the audit, and the evaluation together: a report that reads well with a failed audit is not done. Save the report with its audit and scores attached, so a reader can see how much to trust it.

In [ ]:
state: dict = {"query": REQUEST, "errors": []}
for update in report_graph.stream(state, stream_mode="updates"):
    for node, written in update.items():
        state.update(written or {})
        print(f"[{node}] wrote {list(written or {})}")

report, audit, evaluation, verification = state["report"], state["citation_audit"], state["evaluation"], state["verification"]
display(Markdown(report.report_markdown))
print("citation audit:", audit.model_dump_json())
print("evaluation:", evaluation.model_dump_json())
for i, s in enumerate(verification.approved_sources, 1):
    print(f"[{i}] {s.source_type}: {s.url}")
if state.get("errors"):
    print("recorded partial failures:", state["errors"])

In [ ]:
scores = {k: getattr(evaluation, k) for k in ("coverage", "grounding", "usefulness", "citation_integrity")}
REPORT_MD = "\n".join([
    report.report_markdown.rstrip(), "",
    "## Citation audit", "",
    f"Passed: {audit.passed}. Unknown URLs: {audit.unknown_urls or 'none'}. Duplicates: {audit.duplicate_urls or 'none'}. "
    f"Missing markers: {audit.missing_citation_numbers or 'none'}. Markers without citations: {audit.markers_without_citations or 'none'}.", "",
    "## Evaluation", "",
    "| criterion | score |", "|---|---|", *[f"| {k} | {v} |" for k, v in scores.items()], "",
    f"Passed: {evaluation.passed}. Improvements: " + "; ".join(evaluation.improvements or ["none"]), "",
    "## Approved sources", "", *[f"- {s.url}" for s in verification.approved_sources],
    "", f"Unsupported claims: {len(verification.unsupported_claims)}. Open questions: {len(report.open_questions)}."])
ws.save("multi_agent_report", REPORT_MD)

You should see one line per node, the rendered report with numeric markers and a Sources section, an audit with `passed: true`, the scores, and a ✅ line. Stop here if the audit failed: read which URL or marker it named before you rerun anything.

## Your turn

Add one specialist without giving the supervisor a new low-level tool: a failures specialist that reads only the trajectories. Write its prompt, give it a tool over `ws.load("trajectories")`, wrap it in one delegate tool, and extend `SpecialistName` and the scoper prompt. Explain what should and should not cross its boundary.

In [ ]:
# Shape only. Fill in the prompt and the tool, then add the delegate to DELEGATES and rebuild the supervisor.
FAILURES_PROMPT = f"You are the failures specialist. Today is {TODAY}. ..."


@tool("search_trajectories", description="...")
def search_trajectories(query: str) -> str:
    rows = ws.load("trajectories")
    return f"{len(rows)} trajectories loaded; write the search."


print("fill in FAILURES_PROMPT and search_trajectories, then create the agent and its delegate tool")

# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| Two local tools over your own results | Versioned corpora plus monitored external sources |
| Two or three specialist workers | Larger worker pools with routing and cost policies |
| Pydantic handoff contracts | Versioned schemas with compatibility tests |
| A straight-line graph, no checkpoints | Durable checkpoints, interrupts for human review, resumable jobs |
| A code citation audit | Full provenance ledgers, document snapshots, review workflows |
| One evaluator agent | Eval gates and cost dashboards per release |

## Responsible controls

- Every claim in a generated report cites a source with a stable handle.
- A citation audit that fails the report, not just warns.
- A budget per worker and per run.


## Grow further

- Add a checkpointer and an interrupt after verification, so a reviewer approves the source ledger before the writer runs.
- Add trace assertions: fail if the report cites a URL the verifier never opened, or if it never cites the RAGAS page.
- Run the same request with web research off and on. What became more current, and what became less traceable?